In [0]:
eilerspath='development_042_silver_sandbox.demand_forecast.eilers_landbased_performance_clean_sl'
mappingpath='development_042_silver_sandbox.demand_forecast.eilers_to_gdna_mapping_silver'
atrpath='development_042_silver_sandbox.demand_forecast.igt_everi_attributes_silver'
destpath='development_042_silver_sandbox.demand_forecast.game_attributes_clustering_gold'

In [0]:
eilers_df = spark.read.table(eilerspath)
mapping_df = spark.read.table(mappingpath)
atr_df = spark.read.table(atrpath)

display(eilers_df.limit(5))
display(mapping_df.limit(5))
display(atr_df.limit(5))

In [0]:
from pyspark.sql import functions as F

joined_df = atr_df.join(
    mapping_df,
    atr_df.theme_name == F.lower(mapping_df.bp_theme_name),
    "inner"
)

display(joined_df)

In [0]:
eilers_min_df = (
    eilers_df
    .groupBy("game_name")
    .agg(F.min("game_release_date").alias("min_game_release_date"))
)

result_df = joined_df.join(
    eilers_min_df,
    joined_df.correction == eilers_min_df.game_name,
    "inner"
)

display(result_df)

In [0]:
result_df.write.mode("overwrite").saveAsTable(destpath)